# Parte 3 – Cruce por ubigeo y análisis
Grupo 9 · Temporada: 1 de enero al 31 de mayo de 2022

In [1]:
%pip install plotly nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

nombres = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho", "Cajamarca", "Callao",
           "Cusco", "Huancavelica", "Huánuco", "Ica", "Junín", "La Libertad", "Lambayeque",
           "Lima", "Loreto", "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
           "San Martín", "Tacna", "Tumbes", "Ucayali"]
ubigeos = pd.DataFrame({
    "ubigeo": [f"{i:02d}" for i in range(1, 26)],   # "01", "02", ... "25"
    "departamento": nombres,
})
ubigeos.to_csv("datos/ubigeos.csv", index=False, encoding="utf-8-sig")
ubigeos.head()

,ubigeo,departamento
0,01,Amazonas
1,02,Áncash
2,03,Apurímac
3,04,Arequipa
4,05,Ayacucho


In [3]:
normal = pd.read_csv("datos/ubigeos.csv")
como_texto = pd.read_csv("datos/ubigeos.csv", dtype={"ubigeo": str})

print("Lectura normal:")
print(normal.head(3), "\n", normal.dtypes, "\n")
print("Con dtype={'ubigeo': str}:")
print(como_texto.head(3), "\n", como_texto.dtypes)

Lectura normal:
   ubigeo departamento
0       1     Amazonas
1       2       Áncash
2       3     Apurímac 
 ubigeo          int64
departamento      str
dtype: object 

Con dtype={'ubigeo': str}:
  ubigeo departamento
0     01     Amazonas
1     02       Áncash
2     03     Apurímac 
 ubigeo          str
departamento    str
dtype: object


### ¿Por qué el ubigeo debe leerse como texto?
Con `pd.read_csv` normal, pandas cree que el ubigeo es un número: "01" se convierte en `1` (tipo `int64`) y se pierde el cero inicial. Con `dtype={"ubigeo": str}` se mantiene como texto: "01".
Importa porque el ubigeo es un **código**, no una cantidad: no se suma ni se promedia. Los códigos oficiales del INEI tienen 2 dígitos (y 6 a nivel distrital, como "150101"). Si se pierde el cero, "01" ya no coincide con "01" en otra tabla y el cruce falla, o se confunden códigos.

In [4]:
decretos = pd.read_csv("datos/decretos_por_departamento.csv", encoding="utf-8-sig")
lluvias = pd.read_csv("datos/lluvias_por_departamento.csv", encoding="utf-8-sig")

dec_u = decretos.merge(como_texto, on="departamento", how="left")
llu_u = lluvias.merge(como_texto, on="departamento", how="left")

print("Decretos sin ubigeo:", dec_u.loc[dec_u["ubigeo"].isna(), "departamento"].tolist())
print("Lluvias sin ubigeo:", llu_u.loc[llu_u["ubigeo"].isna(), "departamento"].tolist())

Decretos sin ubigeo: []
Lluvias sin ubigeo: []


### Verificación del ubigeo
Ningún departamento quedó sin ubigeo en ninguna de las dos tablas. Funcionó a la primera porque los nombres coinciden exactamente con los del INEI, tildes incluidas ("Áncash", "Junín", "San Martín"): en la Parte 1 usamos la lista de departamentos del enunciado, y en la Parte 2 limpiamos los nombres de Wikipedia, por ejemplo quitando "[nota 6]" de "Callao". Si hubiéramos dejado "Callao[nota 6]" o escrito "Ancash" sin tilde, ese departamento habría aparecido aquí sin ubigeo.

In [5]:
tabla = llu_u.merge(dec_u[["ubigeo", "declaratorias", "prorrogas"]], on="ubigeo", how="left")
tabla[["declaratorias", "prorrogas"]] = tabla[["declaratorias", "prorrogas"]].fillna(0).astype(int)
tabla = tabla[["ubigeo", "departamento", "capital", "latitud", "longitud",
               "lluvia_total_mm", "dias_lluvia_fuerte", "declaratorias", "prorrogas"]]
tabla = tabla.sort_values("ubigeo").reset_index(drop=True)

print("Filas:", len(tabla))
tabla.to_csv("datos/tabla_final.csv", index=False, encoding="utf-8-sig")

# comprobación: el ubigeo quedó como texto de 2 dígitos
revisar = pd.read_csv("datos/tabla_final.csv", dtype={"ubigeo": str})
print("Primeros ubigeos guardados:", revisar["ubigeo"].head(3).tolist())
tabla

Filas: 25
Primeros ubigeos guardados: ['01', '02', '03']


,ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
0,01,Amazonas,Chachapoyas,-6.23169,-77.86903,436.5,1,1,0
1,02,Áncash,Huaraz,-9.52614,-77.52869,975.3,7,0,0
2,03,Apurímac,Abancay,-13.63426,-72.88380,711.8,2,0,0
3,04,Arequipa,Arequipa,-16.39899,-71.53747,189.9,0,0,0
4,05,Ayacucho,Ayacucho,-13.16380,-74.22345,485.1,0,1,0
5,06,Cajamarca,Cajamarca,-7.16378,-78.50027,773.3,1,0,0
6,07,Callao,Callao,-12.05162,-77.13452,39.2,0,0,0
7,08,Cusco,Cusco,-13.53188,-71.96701,710.7,3,0,0
8,09,Huancavelica,Huancavelica,-12.78691,-74.97298,803.8,1,0,0
9,10,Huánuco,Huánuco,-9.92882,-76.23989,444.8,1,0,0


In [6]:
print("Departamentos con 0 declaratorias:", (tabla["declaratorias"] == 0).sum())
print(tabla.loc[tabla["declaratorias"] > 0, ["ubigeo", "departamento", "lluvia_total_mm", "declaratorias"]])

Departamentos con 0 declaratorias: 22
   ubigeo departamento  lluvia_total_mm  declaratorias
0      01     Amazonas            436.5              1
4      05     Ayacucho            485.1              1
19     20        Piura             89.2              1


In [7]:
import plotly.express as px

fig1 = px.scatter(tabla, x="lluvia_total_mm", y="declaratorias", text="departamento",
                  title="Lluvia total vs. declaratorias por lluvias (ene–may 2022)",
                  labels={"lluvia_total_mm": "Lluvia total en la capital (mm)",
                          "declaratorias": "Número de declaratorias"})
fig1.update_traces(textposition="top center")
fig1.show()

In [8]:
top10 = tabla.sort_values(["declaratorias", "lluvia_total_mm"], ascending=False).head(10)
fig2 = px.bar(top10, x="departamento", y="declaratorias",
              title="Top 10 departamentos con más declaratorias por lluvias (ene–may 2022)",
              labels={"departamento": "Departamento", "declaratorias": "Número de declaratorias"})
fig2.show()

In [9]:
dl = pd.read_csv("datos/decretos_lluvias.csv", encoding="utf-8-sig")
declaratorias = dl[dl["tipo"] == "declara"]
print("Declaratorias de lluvias:", len(declaratorias))
print((declaratorias["motivo"].value_counts(normalize=True) * 100).round(1))

Declaratorias de lluvias: 1
motivo
impacto de daños    100.0
Name: proportion, dtype: float64


## Parte 3: Análisis

### 1. ¿Coinciden los departamentos con más declaratorias y los de más lluvia?
Los 3 departamentos con más declaratorias por lluvias en 2022 son **Amazonas, Ayacucho y Piura** (1 cada uno; el resto tiene 0). Los 3 con más lluvia acumulada (enero a mayo 2022, en su capital) son **Loreto, Ucayali y Áncash**. **No coinciden.** Piura incluso está entre los que menos llovieron (89.2 mm).

### 2. Casos que no cuadran y posibles razones
- **Mucha lluvia y ninguna declaratoria:** Loreto, Ucayali, Áncash, Junín y Madre de Dios.
- **Poca lluvia y con declaratoria:** Piura.

Posibles razones:
1. **La capital no representa al departamento.** La lluvia se midió en un solo punto (la capital), pero los daños pueden haber sido en otras provincias o distritos.
2. **Una declaratoria depende del daño, no del total de lluvia.** Influyen la vulnerabilidad (viviendas, quebradas, huaicos) y la intensidad en pocos días. En la Amazonía llueve mucho todos los años y la población está más adaptada.
3. **El decreto se tramita a pedido del gobierno regional**, así que también depende de que la región lo solicite.
4. **La búsqueda usó "precipitaciones"**, y pueden haber quedado fuera decretos que solo dicen "lluvias".

### 3. Peligro inminente vs. impacto de daños
De las declaratorias por lluvias, **0 % fueron por peligro inminente y 100 % por impacto de daños**. Esto sugiere una gestión del riesgo **reactiva**: el Estado declara la emergencia cuando el daño ya ocurrió y no antes. Pero como hay un solo decreto (n = 1), no se puede generalizar.

### 4. Limitaciones
1. Muestra muy pequeña: solo 1 decreto de lluvias en la temporada 2022.
2. Un solo punto de lluvia por departamento (la capital). Lima y Callao tienen el mismo valor (39.2 mm) porque caen en la misma celda del modelo.
3. Los datos de Open-Meteo son de reanálisis (modelo), no de estaciones meteorológicas.
4. Los departamentos se identificaron solo por el título del decreto. Contamos decretos, no distritos ni población afectada.
5. El término de búsqueda puede no captar todos los decretos.

### 5. Conclusión
En la temporada de lluvias 2022 (enero a mayo) encontramos 29 normas de la PCM con la búsqueda "precipitaciones", pero solo 1 era una declaratoria de emergencia por lluvias (DS 032-2022-PCM, para Amazonas, Ayacucho y Piura). Al cruzarla con la lluvia acumulada de Open-Meteo, los departamentos con declaratoria no son los más lluviosos: Loreto, Ucayali y Áncash tuvieron más lluvia y ninguna declaratoria. Esto indica que la emergencia responde más al daño y a la vulnerabilidad local que al total de lluvia. El único decreto fue por impacto de daños, lo que apunta a una respuesta reactiva. Con una muestra tan pequeña y un solo punto de lluvia por departamento, estos resultados son exploratorios y no concluyentes.